In [5]:
import numpy as np
import matplotlib.pyplot as plt

# Install Qiskit Aer
%pip install qiskit-aer --quiet
from qiskit_aer import AerSimulator

from qiskit.transpiler import Target, InstructionProperties
from qiskit.circuit.library.standard_gates import HGate, SGate, CXGate, IGate, ZGate, XGate, YGate
from qiskit.transpiler import CouplingMap
from qiskit_device_benchmarking.bench_code.mrb import MirrorQA,QuantumAwesomeness
from qiskit.circuit.library.standard_gates import RXGate
import time

# --- Define Clifford + required gates ---
clifford_gates = [IGate(), HGate(), SGate(), CXGate(), ZGate(), XGate(), YGate(), RXGate(np.pi)]


# --- Setup Stabilizer Backend with Expliit Target (only Clifford gates) ---

# --- Parameters ---
n_qubits = 10
shots = 10000
lengths = [2, 4, 10, 20, 50]

# --- Define Clifford gates ---
clifford_gates = [IGate(), HGate(), SGate(), CXGate(), ZGate(), XGate(), YGate(),RXGate(np.pi/2)]

# --- Set up Target safely ---

# --- Set up Target safely ---
target = Target()
for gate in clifford_gates:
    name = gate.name
    num_qubits = gate.num_qubits

    if name in target.instructions:
        continue  # ✅ Skip if already added

    if num_qubits == 1:
        props = {(q,): InstructionProperties() for q in range(n_qubits)}
    elif num_qubits == 2:
        props = {
            (q1, q2): InstructionProperties()
            for q1 in range(n_qubits)
            for q2 in range(n_qubits)
            if q1 != q2
        }
    target.add_instruction(gate, props)


# --- Create stabilizer simulator backend ---
backend = AerSimulator(method="stabilizer", target=target)

# --- Set up experiment ---
exp = MirrorQA(
    range(n_qubits),
    lengths,
    backend=backend,
    two_qubit_gate_density=0.25,
    num_samples=10,
    initial_entangling_angle=np.pi / 2  # stabilizer-safe
)
exp.set_run_options(shots=shots)

# --- Run ---
rb_data = exp.run()


print("✅ Job IDs:", rb_data.job_ids)

# Run the experiment with timing
print("Starting simulation...")
start_time = time.time()
rb_data = exp.run()
end_time = time.time()
print(f"Simulation completed in {end_time - start_time:.2f} seconds")
print("Job IDs:", rb_data.job_ids)


Note: you may need to restart the kernel to use updated packages.
✅ Job IDs: ['f4b48501-a5ad-4cd8-b2cd-0f5791cc1466']
Starting simulation...
Simulation completed in 1.13 seconds
Job IDs: ['2c1f97fb-6dec-4e89-adf2-bf95fac9266f']


In [7]:

# Debug: Inspect rb_data contents
print("Experiment data contents:")
try:
    data_entries = rb_data.data()
    for i, data in enumerate(data_entries):
        print(f"Data entry {i}: {data}")
        # Check required metadata
        required_keys = ["pairs", "singles", "target", "coupling_map"]
        metadata = data.get("metadata", {})
        for key in required_keys:
            print(f"  Metadata '{key}' present: {key in metadata}")
except Exception as e:
    print(f"Error accessing rb_data.data(): {e}")

Provider for ExperimentData object doesn't exist, resulting in a failed attempt to retrieve data from the server; no stored result data exists


Experiment data contents:


In [27]:
print('Pairs:', exp._pairs[0])
exp._static_trans_circuits[0].draw(fold=-1)

Pairs: [(2, 6), (4, 5), (8, 1)]


global phase: 5π/4
         ┌───┐ ░    ┌───┐       ┌───┐                     ░ ┌───┐ ░    ┌───┐    ┌───┐           ░ ┌───┐ ░  ░ ┌─┐                           
    q_0: ┤ X ├─░────┤ H ├───────┤ Z ├─────────────────────░─┤ X ├─░────┤ H ├────┤ X ├───────────░─┤ Y ├─░──░─┤M├───────────────────────────
         ├───┤ ░    └───┘       └───┘    ┌───┐            ░ └───┘ ░    └───┘    └───┘┌───┐      ░ ├───┤ ░  ░ └╥┘┌─┐                        
    q_1: ┤ Z ├─░─────────────────────────┤ X ├────────────░───────░──────────────────┤ X ├──────░─┤ X ├─░──░──╫─┤M├────────────────────────
         ├───┤ ░             ┌─────────┐ └─┬─┘            ░ ┌───┐ ░                  └─┬─┘      ░ ├───┤ ░  ░  ║ └╥┘┌─┐                     
    q_2: ┤ Z ├─░──────■──────┤ Rx(π/2) ├───┼──────────────░─┤ X ├─░──────■─────────────┼────────░─┤ Z ├─░──░──╫──╫─┤M├─────────────────────
         ├───┤ ░      │      └──┬───┬──┘   │     ┌───┐    ░ ├───┤ ░      │      ┌───┐  │  ┌───┐ ░ ├───┤ ░  ░  ║  ║ └╥┘┌─┐                  
    q_3: ┤ Y ├─░──────┼─────────┤ S ├──────┼─────┤ Y ├────░─┤ X ├─░──────┼──────┤ S ├──┼──┤ Y ├─░─┤ Z ├─░──░──╫──╫──╫─┤M├──────────────────
         └───┘ ░      │         └───┘      │  ┌──┴───┴──┐ ░ ├───┤ ░      │      └───┘  │  └───┘ ░ ├───┤ ░  ░  ║  ║  ║ └╥┘┌─┐               
    q_4: ──────░──────┼───────────■────────┼──┤ Rx(π/2) ├─░─┤ Y ├─░──────┼────────■────┼────────░─┤ Y ├─░──░──╫──╫──╫──╫─┤M├───────────────
               ░      │         ┌─┴─┐      │  └─────────┘ ░ └───┘ ░      │      ┌─┴─┐  │        ░ ├───┤ ░  ░  ║  ║  ║  ║ └╥┘┌─┐            
    q_5: ──────░──────┼─────────┤ X ├──────┼──────────────░───────░──────┼──────┤ X ├──┼────────░─┤ Y ├─░──░──╫──╫──╫──╫──╫─┤M├────────────
         ┌───┐ ░    ┌─┴─┐       └───┘      │              ░ ┌───┐ ░    ┌─┴─┐    └───┘  │        ░ └───┘ ░  ░  ║  ║  ║  ║  ║ └╥┘┌─┐         
    q_6: ┤ Y ├─░────┤ X ├──────────────────┼──────────────░─┤ X ├─░────┤ X ├───────────┼────────░───────░──░──╫──╫──╫──╫──╫──╫─┤M├─────────
         └───┘ ░    ├───┤    ┌──────────┐  │              ░ ├───┤ ░    ├───┤    ┌───┐  │        ░ ┌───┐ ░  ░  ║  ║  ║  ║  ║  ║ └╥┘┌─┐      
    q_7: ──────░────┤ H ├────┤ Rx(-π/2) ├──┼──────────────░─┤ Z ├─░────┤ H ├────┤ S ├──┼────────░─┤ X ├─░──░──╫──╫──╫──╫──╫──╫──╫─┤M├──────
               ░    └───┘    └──────────┘  │  ┌─────────┐ ░ ├───┤ ░    └───┘    └───┘  │        ░ ├───┤ ░  ░  ║  ║  ║  ║  ║  ║  ║ └╥┘┌─┐   
    q_8: ──────░───────────────────────────■──┤ Rx(π/2) ├─░─┤ X ├─░────────────────────■────────░─┤ X ├─░──░──╫──╫──╫──╫──╫──╫──╫──╫─┤M├───
         ┌───┐ ░ ┌──────────┐   ┌───┐         └─────────┘ ░ ├───┤ ░ ┌──────────┐┌───┐           ░ ├───┤ ░  ░  ║  ║  ║  ║  ║  ║  ║  ║ └╥┘┌─┐
    q_9: ┤ X ├─░─┤ Rx(-π/2) ├───┤ Z ├─────────────────────░─┤ Z ├─░─┤ Rx(-π/2) ├┤ Z ├───────────░─┤ Z ├─░──░──╫──╫──╫──╫──╫──╫──╫──╫──╫─┤M├
         └───┘ ░ └──────────┘   └───┘                     ░ └───┘ ░ └──────────┘└───┘           ░ └───┘ ░  ░  ║  ║  ║  ║  ║  ║  ║  ║  ║ └╥┘
meas: 10/═════════════════════════════════════════════════════════════════════════════════════════════════════╩══╩══╩══╩══╩══╩══╩══╩══╩══╩═
                                                                                                              0  1  2  3  4  5  6  7  8  9

In [28]:
# Count operations before transpilation for circuit 0
print("Operations before transpilation:")
original_circuits = exp.circuits()
print(original_circuits[0].count_ops())
# Count 1Q and 2Q for each circuit before transpilation
print("1Q and 2Q gates before transpilation for all circuits:")
for i, circ in enumerate(original_circuits):
    total_1Q_gates = 0
    total_2Q_gates = 0
    for gate, count in circ.count_ops().items():
        if gate in ['cx']:
            total_2Q_gates += count
        else:
            total_1Q_gates += count
    print(f"Circuit {i}: Total 1Q gates: {total_1Q_gates}, Total 2Q gates: {total_2Q_gates}")
# Count operations after transpilation for circuit 0
print("Operations after transpilation:")
exp._static_trans_circuits[0].count_ops()
# Count 1Q and 2Q for each circuit after transpilation
print("1Q and 2Q gates after transpilation for all circuits:")
for i, circ in enumerate(exp._static_trans_circuits):
    total_1Q_gate_trans = 0
    total_2Q_gate_trans = 0
    for gate, count in circ.count_ops().items():
        if gate in ['cx']:
            total_2Q_gate_trans += count
        else:
            total_1Q_gate_trans += count
    print(f"Circuit {i} transpiled: Total 1Q gates: {total_1Q_gate_trans}, Total 2Q gates: {total_2Q_gate_trans}")

Operations before transpilation:
OrderedDict({'Clifford-1Q(12)': 11, 'measure': 10, 'Clifford-1Q(0)': 9, 'Clifford-1Q(18)': 8, 'barrier': 6, 'cx': 6, 'Clifford-1Q(6)': 4, 'rx': 3, 'Clifford-1Q(11)': 2, 'Clifford-1Q(15)': 2, 'Clifford-1Q(4)': 1, 'Clifford-1Q(22)': 1})
1Q and 2Q gates before transpilation for all circuits:
Circuit 0: Total 1Q gates: 57, Total 2Q gates: 6
Circuit 1: Total 1Q gates: 83, Total 2Q gates: 16
Circuit 2: Total 1Q gates: 186, Total 2Q gates: 28
Circuit 3: Total 1Q gates: 362, Total 2Q gates: 52
Circuit 4: Total 1Q gates: 880, Total 2Q gates: 122
Circuit 5: Total 1Q gates: 60, Total 2Q gates: 4
Circuit 6: Total 1Q gates: 90, Total 2Q gates: 12
Circuit 7: Total 1Q gates: 185, Total 2Q gates: 30
Circuit 8: Total 1Q gates: 358, Total 2Q gates: 54
Circuit 9: Total 1Q gates: 858, Total 2Q gates: 134
Circuit 10: Total 1Q gates: 57, Total 2Q gates: 6
Circuit 11: Total 1Q gates: 86, Total 2Q gates: 14
Circuit 12: Total 1Q gates: 180, Total 2Q gates: 32
Circuit 13: Total 

In [1]:
exp.analysis.set_options(analyzed_quantity='Effective Polarization')
#exp.analysis.set_options(analyzed_quantity='Mutual Information')
analysis = exp.analysis.run(rb_data)

NameError: name 'exp' is not defined

In [ ]:
analysis.

In [30]:
print(rb_data.status())
print(rb_data.figure_names)


ExperimentStatus.ERROR
[]


In [31]:
analysis.figure(0)

ExperimentEntryNotFound: 'Figure index 0 out of range.'

In [39]:
import numpy as np
from qiskit.transpiler import Target, InstructionProperties
from qiskit.circuit.library.standard_gates import HGate, SGate, CXGate, IGate, ZGate, XGate, YGate, RZGate
from qiskit_aer.noise import NoiseModel, depolarizing_error
from qiskit_aer import AerSimulator
from qiskit_device_benchmarking.bench_code.mrb import MirrorQA,QuantumAwesomeness
from qiskit.circuit.library.standard_gates import RXGate


n_qubits = 25
p1 = 0.001
p2 = 0.01
shots = 10000
lengths = [2, 4, 10, 20, 50]
clifford_gates = [IGate(), HGate(), SGate(), CXGate(), ZGate(), XGate(), YGate(), RXGate(np.pi/2)]

target = Target()
for gate in clifford_gates:
    name = gate.name
    num_qubits = gate.num_qubits
    if name in target.instructions:
        continue
    if num_qubits == 1:
        props = {(q,): InstructionProperties(error=p1) for q in range(n_qubits)}
    elif num_qubits == 2:
        props = {(q1, q2): InstructionProperties(error=p2)
                 for q1 in range(n_qubits)
                 for q2 in range(n_qubits) if q1 != q2}
    target.add_instruction(gate, props)

noise_model = NoiseModel()
for gate in ["id", "h", "x", "y", "z", "rx", "s", "sdg"]:
    noise_model.add_all_qubit_quantum_error(depolarizing_error(p1, 1), [gate])
noise_model.add_all_qubit_quantum_error(depolarizing_error(p2, 2), ["cx"])

backend = AerSimulator(method="stabilizer", target=target, noise_model=noise_model)

exp = MirrorQA(
    range(n_qubits),
    lengths,
    backend=backend,
    two_qubit_gate_density=0.25,
    num_samples=10,
    initial_entangling_angle=np.pi / 2  # stabilizer-safe
)
exp.set_run_options(shots=shots)

# --- Run ---
rb_data = exp.run()


print("✅ Job IDs:", rb_data.job_ids)


✅ Job IDs: ['be6da318-01a3-4b7d-a54b-7a23b1d569bb']


In [40]:
print('Pairs:', exp._pairs[0])
exp._static_trans_circuits[0].draw(fold=-1)

Pairs: [(22, 10), (20, 9), (3, 7), (2, 15), (8, 5), (19, 1), (18, 6)]


global phase: 2π
               ░    ┌───┐    ┌──────────┐   ┌───┐                                                  ░ ┌───┐ ░    ┌───┐    ┌───┐┌───┐                                 ░ ┌───┐ ░  ░ ┌─┐                                                                        
    q_0: ──────░────┤ H ├────┤ Rx(-π/2) ├───┤ X ├──────────────────────────────────────────────────░─┤ Z ├─░────┤ H ├────┤ S ├┤ Z ├─────────────────────────────────░─┤ Z ├─░──░─┤M├────────────────────────────────────────────────────────────────────────
         ┌───┐ ░    └───┘    └──────────┘   └───┘   ┌───┐                                          ░ ├───┤ ░    └───┘    └───┘└───┘┌───┐                            ░ ├───┤ ░  ░ └╥┘┌─┐                                                                     
    q_1: ┤ X ├─░────────────────────────────────────┤ X ├──────────────────────────────────────────░─┤ Z ├─░───────────────────────┤ X ├────────────────────────────░─┤ Z ├─░──░──╫─┤M├─────────────────────────────────────────────────────────────────────
         └───┘ ░                                    └─┬─┘┌─────────┐                               ░ ├───┤ ░                       └─┬─┘                            ░ └───┘ ░  ░  ║ └╥┘┌─┐                                                                  
    q_2: ──────░──────────────────────────────■───────┼──┤ Rx(π/2) ├───────────────────────────────░─┤ X ├─░────────────────────■────┼──────────────────────────────░───────░──░──╫──╫─┤M├──────────────────────────────────────────────────────────────────
         ┌───┐ ░             ┌─────────┐      │       │  └─────────┘                               ░ └───┘ ░                    │    │                              ░ ┌───┐ ░  ░  ║  ║ └╥┘┌─┐                                                               
    q_3: ┤ X ├─░──────■──────┤ Rx(π/2) ├──────┼───────┼────────────────────────────────────────────░───────░──────■─────────────┼────┼──────────────────────────────░─┤ Y ├─░──░──╫──╫──╫─┤M├───────────────────────────────────────────────────────────────
         ├───┤ ░      │      └──┬───┬──┘      │       │  ┌──────────┐                              ░ ┌───┐ ░      │      ┌───┐  │    │  ┌───┐                       ░ ├───┤ ░  ░  ║  ║  ║ └╥┘┌─┐                                                            
    q_4: ┤ Y ├─░──────┼─────────┤ H ├─────────┼───────┼──┤ Rx(-π/2) ├──────────────────────────────░─┤ X ├─░──────┼──────┤ H ├──┼────┼──┤ S ├───────────────────────░─┤ Z ├─░──░──╫──╫──╫──╫─┤M├────────────────────────────────────────────────────────────
         └───┘ ░      │         ├───┤         │       │  └──────────┘                              ░ ├───┤ ░      │      ├───┤  │    │  └───┘                       ░ ├───┤ ░  ░  ║  ║  ║  ║ └╥┘┌─┐                                                         
    q_5: ──────░──────┼─────────┤ X ├─────────┼───────┼────────────────────────────────────────────░─┤ Z ├─░──────┼──────┤ X ├──┼────┼──────────────────────────────░─┤ X ├─░──░──╫──╫──╫──╫──╫─┤M├─────────────────────────────────────────────────────────
               ░      │         └─┬─┘         │       │     ┌───┐                                  ░ └───┘ ░      │      └─┬─┘  │    │  ┌───┐                       ░ ├───┤ ░  ░  ║  ║  ║  ║  ║ └╥┘┌─┐                                                      
    q_6: ──────░──────┼───────────┼───────────┼───────┼─────┤ X ├──────────────────────────────────░───────░──────┼────────┼────┼────┼──┤ X ├───────────────────────░─┤ Z ├─░──░──╫──╫──╫──╫──╫──╫─┤M├──────────────────────────────────────────────────────
         ┌───┐ ░    ┌─┴─┐         │           │       │     └─┬─┘                                  ░ ┌───┐ ░    ┌─┴─┐      │    │    │  └─┬─┘                       ░ ├───┤ ░  ░  ║  ║  ║  ║  ║  ║ └╥┘┌─┐                                                   
    q_7: ┤ Z ├─░────┤ X ├─────────┼───────────┼───────┼───────┼────────────────────────────────────░─┤ X ├─░────┤ X ├──────┼────┼────┼────┼─────────────────────────░─┤ Y ├─░──░──╫──╫──╫──╫

In [41]:
print('Pairs:', exp._pairs[0])
exp._static_trans_circuits[0].draw(fold=0)

Pairs: [(22, 10), (20, 9), (3, 7), (2, 15), (8, 5), (19, 1), (18, 6)]


global phase: 2π
               ░    ┌───┐    ┌──────────┐   ┌───┐                    »
    q_0: ──────░────┤ H ├────┤ Rx(-π/2) ├───┤ X ├────────────────────»
         ┌───┐ ░    └───┘    └──────────┘   └───┘   ┌───┐            »
    q_1: ┤ X ├─░────────────────────────────────────┤ X ├────────────»
         └───┘ ░                                    └─┬─┘┌─────────┐ »
    q_2: ──────░──────────────────────────────■───────┼──┤ Rx(π/2) ├─»
         ┌───┐ ░             ┌─────────┐      │       │  └─────────┘ »
    q_3: ┤ X ├─░──────■──────┤ Rx(π/2) ├──────┼───────┼──────────────»
         ├───┤ ░      │      └──┬───┬──┘      │       │  ┌──────────┐»
    q_4: ┤ Y ├─░──────┼─────────┤ H ├─────────┼───────┼──┤ Rx(-π/2) ├»
         └───┘ ░      │         ├───┤         │       │  └──────────┘»
    q_5: ──────░──────┼─────────┤ X ├─────────┼───────┼──────────────»
               ░      │         └─┬─┘         │       │     ┌───┐    »
    q_6: ──────░──────┼───────────┼───────────┼───────┼─────┤ X ├────»
         ┌───┐ ░    ┌─┴─┐         │           │       │     └─┬─┘    »
    q_7: ┤ Z ├─░────┤ X ├─────────┼───────────┼───────┼───────┼──────»
         ├───┤ ░    └───┘         │           │       │       │      »
    q_8: ┤ X ├─░──────────────────■───────────┼───────┼───────┼──────»
         ├───┤ ░                ┌───┐         │       │       │      »
    q_9: ┤ Y ├─░────────────────┤ X ├─────────┼───────┼───────┼──────»
         ├───┤ ░    ┌───┐       └─┬─┘         │       │       │      »
   q_10: ┤ Y ├─░────┤ X ├─────────┼───────────┼───────┼───────┼──────»
         ├───┤ ░    └─┬─┘         │           │       │       │      »
   q_11: ┤ Z ├─░──────┼───────────┼───────────┼───────┼───────┼──────»
         ├───┤ ░      │           │           │       │       │      »
   q_12: ┤ Z ├─░──────┼───────────┼───────────┼───────┼───────┼──────»
         ├───┤ ░      │           │           │       │       │      »
   q_13: ┤ Y ├─░──────┼───────────┼───────────┼───────┼───────┼──────»
         ├───┤ ░      │           │           │       │       │      »
   q_14: ┤ Z ├─░──────┼───────────┼───────────┼───────┼───────┼──────»
         ├───┤ ░      │           │         ┌─┴─┐     │       │      »
   q_15: ┤ Z ├─░──────┼───────────┼─────────┤ X ├─────┼───────┼──────»
         └───┘ ░      │           │         ├───┤     │       │      »
   q_16: ──────░──────┼───────────┼─────────┤ H ├─────┼───────┼──────»
         ┌───┐ ░      │           │         ├───┤     │       │      »
   q_17: ┤ Z ├─░──────┼───────────┼─────────┤ S ├─────┼───────┼──────»
         └───┘ ░      │           │         └───┘     │       │      »
   q_18: ──────░──────┼───────────┼───────────────────┼───────■──────»
         ┌───┐ ░      │           │                   │  ┌─────────┐ »
   q_19: ┤ X ├─░──────┼───────────┼───────────────────■──┤ Rx(π/2) ├─»
         ├───┤ ░      │           │      ┌─────────┐     └─────────┘ »
   q_20: ┤ Z ├─░──────┼───────────■──────┤ Rx(π/2) ├─────────────────»
         ├───┤ ░      │         ┌───┐    └─────────┘                 »
   q_21: ┤ Y ├─░──────┼─────────┤ Y ├────────────────────────────────»
         └───┘ ░      │      ┌──┴───┴──┐                             »
   q_22: ──────░──────■──────┤ Rx(π/2) ├─────────────────────────────»
               ░    ┌───┐    └─────────┘                             »
   q_23: ──────░────┤ Z ├────────────────────────────────────────────»
         ┌───┐ ░ ┌──┴───┴───┐   ┌───┐                                »
   q_24: ┤ Z ├─░─┤ Rx(-π/2) ├───┤ X ├────────────────────────────────»
         └───┘ ░ └──────────┘   └───┘                                »
meas: 25/════════════════════════════════════════════════════════════»
                                                                     »
«                                       ░ ┌───┐ ░    ┌───┐    ┌───┐┌───┐     »
«    q_0: ──────────────────────────────░─┤ Z ├─░────┤ H ├────┤ S ├┤ Z ├─────»
«                                       ░ ├───┤ ░    └───┘    

In [42]:
# Count operations before transpilation for circuit 0
print("Operations before transpilation:")
original_circuits = exp.circuits()
print(original_circuits[0].count_ops())
# Count 1Q and 2Q for each circuit before transpilation
print("1Q and 2Q gates before transpilation for all circuits:")
for i, circ in enumerate(original_circuits):
    total_1Q_gates = 0
    total_2Q_gates = 0
    for gate, count in circ.count_ops().items():
        if gate in ['cx']:
            total_2Q_gates += count
        else:
            total_1Q_gates += count
    print(f"Circuit {i}: Total 1Q gates: {total_1Q_gates}, Total 2Q gates: {total_2Q_gates}")
# Count operations after transpilation for circuit 0
print("Operations after transpilation:")
exp._static_trans_circuits[0].count_ops()
# Count 1Q and 2Q for each circuit after transpilation
print("1Q and 2Q gates after transpilation for all circuits:")
for i, circ in enumerate(exp._static_trans_circuits):
    total_1Q_gate_trans = 0
    total_2Q_gate_trans = 0
    for gate, count in circ.count_ops().items():
        if gate in ['cx']:
            total_2Q_gate_trans += count
        else:
            total_1Q_gate_trans += count
    print(f"Circuit {i} transpiled: Total 1Q gates: {total_1Q_gate_trans}, Total 2Q gates: {total_2Q_gate_trans}")


Operations before transpilation:
OrderedDict({'measure': 25, 'Clifford-1Q(12)': 24, 'Clifford-1Q(0)': 21, 'Clifford-1Q(6)': 17, 'Clifford-1Q(18)': 15, 'cx': 10, 'barrier': 6, 'rx': 5, 'Clifford-1Q(17)': 4, 'Clifford-1Q(21)': 4, 'Clifford-1Q(22)': 4, 'Clifford-1Q(4)': 4, 'Clifford-1Q(20)': 2, 'Clifford-1Q(2)': 2, 'Clifford-1Q(8)': 2, 'Clifford-1Q(23)': 1, 'Clifford-1Q(19)': 1, 'Clifford-1Q(5)': 1, 'Clifford-1Q(9)': 1, 'Clifford-1Q(7)': 1, 'Clifford-1Q(3)': 1})
1Q and 2Q gates before transpilation for all circuits:
Circuit 0: Total 1Q gates: 141, Total 2Q gates: 10
Circuit 1: Total 1Q gates: 219, Total 2Q gates: 24
Circuit 2: Total 1Q gates: 465, Total 2Q gates: 56
Circuit 3: Total 1Q gates: 877, Total 2Q gates: 110
Circuit 4: Total 1Q gates: 2035, Total 2Q gates: 312
Circuit 5: Total 1Q gates: 138, Total 2Q gates: 12
Circuit 6: Total 1Q gates: 209, Total 2Q gates: 30
Circuit 7: Total 1Q gates: 445, Total 2Q gates: 68
Circuit 8: Total 1Q gates: 858, Total 2Q gates: 120
Circuit 9: Total 1

In [44]:
print(rb_data.status())
print(rb_data.figure_names)


ExperimentStatus.ERROR
[]


In [43]:
analysis.figure(0)

ExperimentEntryNotFound: 'Figure index 0 out of range.'